# Lab 1 — Profiling LLM Inference Across Hardware

**Course**: *Dataflow Architectures for AI: From Principles to Practice*
**Week**: 1 — The Accelerator Landscape
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research / Columbia Engineering)
**Runtime**: Google Colab, GPU runtime required (T4 is sufficient)
**Expected time**: 90–120 minutes

---

## Learning objectives

By the end of this lab you will be able to:
1. Build a repeatable benchmarking harness using `torch.utils.benchmark` that measures inference latency and throughput for an LLM on CPU and GPU.
2. Profile memory allocation patterns during inference using `torch.profiler` and `torch.cuda.memory_allocated`.
3. Construct the first row of the course **Accelerator Landscape Map** from *measurements you made yourself*, not vendor slides.
4. Explain the measured CPU vs GPU performance gap in terms of specific architectural features (SIMT parallelism, HBM bandwidth, cache hierarchy).

## Prerequisites

- Intermediate Python; comfort with PyTorch `nn.Module` and tensor operations.
- You've read/watched the Week 1 lecture material on control-flow vs dataflow execution and the Accelerator Landscape Map.

## How this lab is graded

This lab has two gradable components:
- **Auto-graded checkpoints (60%)** — assertion cells inside each Part verify your numerical outputs are within expected ranges. These run automatically.
- **Peer-reviewed analysis (40%)** — the final section asks you to write structured paragraphs explaining *why* your measurements came out the way they did. Your peers will grade these against the rubric in `README.md`.

## Submission

When you finish:
1. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO` and answer every `YOUR ANALYSIS HERE` prompt.
3. Run `File → Download .ipynb` and submit via the course portal.


## Part 0 — Environment setup

We will use **Qwen 2.5 0.5B Instruct** throughout. It is small enough for Colab's free T4 GPU and representative enough of modern decoder-only LLM inference to make the measurements meaningful.

> **Why Qwen 2.5 0.5B?** It uses Grouped-Query Attention (GQA), RoPE positional embeddings, and SwiGLU — the three architectural choices that dominate modern open-weight LLMs. Whatever you learn here transfers to Llama 3, Mistral, Qwen 3, and the next model that comes out next quarter.


In [ ]:
# Install dependencies. Colab already has torch; we just need transformers and accelerate.
!pip install -q transformers==4.44.2 accelerate==0.33.0


In [ ]:
import os
import time
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils import benchmark as torch_bench
from torch.profiler import profile, record_function, ProfilerActivity

# Make outputs reproducible
torch.manual_seed(0)

# Quick sanity: GPU runtime is required for this lab.
assert torch.cuda.is_available(), (
    "GPU runtime required. In Colab: Runtime -> Change runtime type -> T4 GPU."
)

print(f"torch       = {torch.__version__}")
print(f"CUDA        = {torch.version.cuda}")
print(f"GPU         = {torch.cuda.get_device_name(0)}")
print(f"VRAM total  = {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer + model in fp16 (inference precision on most modern GPUs)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_gpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()
model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).cpu().eval()

# Build a fixed input. Using a fixed input keeps the benchmark deterministic.
PROMPT = "Explain how a GPU differs from a dataflow accelerator in two sentences."
BATCH_SIZE = 1
SEQ_LEN = 64

inputs = tokenizer([PROMPT] * BATCH_SIZE, return_tensors="pt", padding="max_length",
                   truncation=True, max_length=SEQ_LEN)

inputs_gpu = {k: v.cuda() for k, v in inputs.items()}
inputs_cpu = {k: v.cpu() for k, v in inputs.items()}

# Print model size
n_params = sum(p.numel() for p in model_gpu.parameters())
print(f"Model: {MODEL_ID}")
print(f"Params: {n_params/1e6:.1f} M")
print(f"Input shape: {inputs['input_ids'].shape}")


## Part 1 — Build a benchmarking harness

`torch.utils.benchmark.Timer` is the right tool for this: it handles GPU synchronization, warm-up, and variance estimation automatically. The naive `time.time()` around a forward pass will report *kernel launch time*, not *execution time*, because CUDA calls are asynchronous by default.

**Your task**: write a `bench_forward(model, inputs, label)` function that:
1. Warms up the model (run forward 3 times, discard results)
2. Uses `torch_bench.Timer` to measure a single forward pass
3. Returns the **median** latency in milliseconds

The `Timer` object's `.blocked_autorange()` method is what you want — it runs enough iterations to hit a stable measurement.


In [ ]:
def bench_forward(model, inputs, label, min_run_time=1.0):
    """Measure median forward-pass latency (ms) using torch.utils.benchmark.

    Args:
        model: the nn.Module to benchmark (must already be in eval mode and on-device).
        inputs: dict of input tensors matching model signature (on the right device).
        label: string label for the measurement (used for display).
        min_run_time: minimum seconds of measurement (Timer will iterate until this is met).

    Returns:
        median_latency_ms: float, the measured median forward-pass latency in ms.
    """
    # TODO 1.1 — warm up the model with 3 forward passes. Use torch.no_grad().
    #             Why do we warm up? Because the first few forwards trigger lazy
    #             initialization, kernel compilation, and cache warming — none of
    #             which we want to measure.
    pass  # REPLACE

    # TODO 1.2 — build a torch_bench.Timer.
    #             The stmt should be a string that calls model(**inputs).
    #             Pass `globals={"model": model, "inputs": inputs}` so the Timer
    #             can see the local variables.
    #             Pass label=label and sub_label="forward".
    timer = None  # REPLACE

    # TODO 1.3 — run timer.blocked_autorange(min_run_time=min_run_time) and return
    #             the median in milliseconds.
    #             Note: the returned Measurement has `.median` in seconds.
    raise NotImplementedError("Remove this once you have implemented the function")


In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
# Validates your bench_forward() function on a trivial model.
class _TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(128, 128)
    def forward(self, input):
        return self.fc(input)

_check_model  = _TinyModel().cuda().eval()
_check_inputs = {"input": torch.randn(4, 128, device="cuda")}

_ms = bench_forward(_check_model, _check_inputs, "tiny-check", min_run_time=0.3)
assert _ms is not None, "bench_forward returned None — did you return the median?"
assert 0.001 < _ms < 50.0, f"Reference linear bench looks off: {_ms} ms"
print(f"[OK] bench_forward works. Reference tiny model: {_ms:.3f} ms")


## Part 2 — Run inference on CPU and GPU

Now use your harness to measure the same forward pass on CPU and GPU.

**Your task**:
1. Call `bench_forward` on `model_cpu` with `inputs_cpu`.
2. Call `bench_forward` on `model_gpu` with `inputs_gpu`.
3. Compute **tokens per second** for each: `throughput = (batch_size * seq_len) / (latency_ms / 1000)`.
4. Compute the **speedup** (GPU throughput / CPU throughput).

Record the numbers in the dict `perf`.


In [ ]:
perf = {}

# TODO 2.1 — measure CPU latency and throughput
#             (CPU inference is slow; we pass min_run_time=0.5 to keep the lab short)
cpu_ms = None  # REPLACE
cpu_tps = None  # REPLACE: tokens per second

# TODO 2.2 — measure GPU latency and throughput
gpu_ms = None  # REPLACE
gpu_tps = None  # REPLACE

# TODO 2.3 — compute speedup as GPU_tps / CPU_tps
speedup = None  # REPLACE

perf["cpu_ms"] = cpu_ms
perf["gpu_ms"] = gpu_ms
perf["cpu_tps"] = cpu_tps
perf["gpu_tps"] = gpu_tps
perf["speedup"] = speedup

print(f"CPU:    {cpu_ms:7.1f} ms/fwd   {cpu_tps:7.1f} tok/s")
print(f"GPU:    {gpu_ms:7.1f} ms/fwd   {gpu_tps:7.1f} tok/s")
print(f"Speedup (GPU / CPU): {speedup:.1f}x")


In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
# Sanity checks on your measurements. Ranges below are deliberately wide because
# Colab hardware varies — we just want to catch obvious mistakes.
assert perf["cpu_ms"] > 50,    f"CPU latency suspiciously low: {perf['cpu_ms']} ms"
assert perf["gpu_ms"] < 200,   f"GPU latency suspiciously high: {perf['gpu_ms']} ms"
assert perf["cpu_ms"] > perf["gpu_ms"], "CPU should be slower than GPU here."
assert perf["speedup"] > 2,    f"Expected GPU speedup > 2x on this workload, got {perf['speedup']:.1f}x"
print(f"[OK] Measurements look reasonable. Speedup = {perf['speedup']:.1f}x")


## Part 3 — Profile memory allocation

Latency is half the story. On accelerators, *where and when data moves* is usually the other half. For this part you'll:

1. Measure **peak GPU memory** during a forward pass using `torch.cuda.max_memory_allocated`.
2. Use `torch.profiler` with `profile_memory=True` to capture the memory timeline of one forward pass.
3. Identify the top-3 operators by CUDA time.

This is also your first look at how a **cache-based** memory hierarchy (GPU) behaves. In later labs you'll contrast this with **software-managed** scratchpads (TPU, Spyre).


In [ ]:
# TODO 3.1 — reset peak memory counter, run a forward pass, record peak.
#             Functions you need: torch.cuda.reset_peak_memory_stats(),
#                                 torch.cuda.max_memory_allocated().
#             Remember to torch.cuda.synchronize() after the forward.
peak_bytes = None  # REPLACE

print(f"Peak GPU memory for a single forward: {peak_bytes / 1e6:.1f} MB")

# TODO 3.2 — compute the "memory overhead ratio":
#             peak_bytes divided by the total size of the fp16 model weights.
#             Weights bytes = n_params * 2  (fp16 = 2 bytes)
mem_overhead_ratio = None  # REPLACE
print(f"Memory overhead ratio (peak / weights): {mem_overhead_ratio:.2f}x")


In [ ]:
# Use torch.profiler to capture the op-level memory timeline.
# Note: profile_memory adds overhead — we are not trying to measure latency here.
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
) as prof:
    with torch.no_grad():
        with record_function("llm_forward"):
            _ = model_gpu(**inputs_gpu)
    torch.cuda.synchronize()

# Print top 10 operators by CUDA time.
print(prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=10,
))


In [ ]:
# TODO 3.3 — extract the top-3 operators by *self* CUDA time from the profiler.
#             prof.key_averages() returns a list of FunctionEventAvg objects.
#             Each has .key (op name) and .self_cuda_time_total (microseconds).
#             Sort descending, take top 3, store as (name, time_us) tuples.
top3 = None  # REPLACE  -- should be List[Tuple[str, float]]

for name, t_us in top3:
    print(f"  {name:40s}  {t_us/1000:8.2f} ms")


In [ ]:
# --- Checkpoint 3 -------------------------------------------------------
assert peak_bytes > n_params * 2, "Peak memory should exceed just the model weights."
assert mem_overhead_ratio >= 1.0, f"Overhead ratio should be >= 1.0, got {mem_overhead_ratio}"
assert len(top3) == 3, "top3 must contain exactly 3 operators."
assert all(isinstance(t, tuple) and len(t) == 2 for t in top3), "top3 entries must be (name, time_us) tuples."
print("[OK] Memory profile captured, top-3 operators extracted.")


## Part 4 — Build your first row of the Accelerator Landscape Map

The course's **Accelerator Landscape Map** is not vendor marketing — it's a table you build from measurement. In this lab you populate the *Compute & Memory* row for **GPU (NVIDIA T4 or whatever Colab gave you)** from the numbers you collected.

Fill in the `landscape_row` dict below using your measurements from Parts 1–3.


In [ ]:
# TODO 4.1 — fill in the Landscape Map row for the GPU you measured.
landscape_row = {
    "accelerator":          None,  # e.g. f"GPU ({torch.cuda.get_device_name(0)})"
    "execution_model":      None,  # one short phrase, e.g. "SIMT / control-flow"
    "memory_hierarchy":     None,  # one short phrase, e.g. "L1/L2/HBM (hw-managed cache)"
    "programming_model":    None,  # one short phrase, e.g. "CUDA / PyTorch eager"
    "forward_latency_ms":   None,  # from perf["gpu_ms"]
    "throughput_tps":       None,  # from perf["gpu_tps"]
    "peak_memory_MB":       None,  # from peak_bytes / 1e6
    "dominant_op":          None,  # top3[0][0]
}

for k, v in landscape_row.items():
    print(f"  {k:22s}: {v}")


In [ ]:
# --- Checkpoint 4 -------------------------------------------------------
required_keys = {
    "accelerator", "execution_model", "memory_hierarchy", "programming_model",
    "forward_latency_ms", "throughput_tps", "peak_memory_MB", "dominant_op",
}
assert set(landscape_row.keys()) == required_keys, "Missing or extra keys in landscape_row."
assert all(v is not None for v in landscape_row.values()), "All landscape_row fields must be filled."
assert isinstance(landscape_row["forward_latency_ms"], (int, float))
assert isinstance(landscape_row["throughput_tps"],    (int, float))
print("[OK] Landscape Map row complete. You have contributed the first row!")


## Part 5 — Written analysis (peer-reviewed)

This is the part your peers will grade against the rubric. Be specific. Cite numbers you measured.

Each prompt is worth 10 points; the rubric is in `README.md`. Write 3–6 sentences per answer. Vague or slogan-level answers will not score well.

---

### 5.1 — Explain the measured GPU speedup over CPU in terms of *specific* architectural features.

You measured roughly a `[YOUR_SPEEDUP]x` speedup. Name at least three architectural features that contribute to this gap and explain the contribution of each. (SIMT, HBM bandwidth, cache hierarchy, thousands of parallel threads, dedicated matmul units, low-precision arithmetic — pick three and *explain* them.)

**YOUR ANALYSIS HERE**

---

### 5.2 — Why is the peak memory larger than just the model weights?

Your overhead ratio was roughly `[YOUR_RATIO]x`. Where does the extra memory go during a forward pass? Be specific about *which* tensors are live simultaneously for a 0.5B-parameter decoder-only transformer with GQA.

**YOUR ANALYSIS HERE**

---

### 5.3 — Comparative prediction (the Landscape Map question).

Predict how the **memory overhead ratio** would likely differ on a **dataflow accelerator with software-managed scratchpads** (e.g., Spyre, TPU) vs the cache-based GPU you just measured. Would it be higher, lower, roughly the same? Justify with reference to how scratchpads manage live tensors.

You will revisit this prediction in Week 4. It's okay to be wrong — but be *concrete* about why you're predicting what you're predicting.

**YOUR ANALYSIS HERE**


In [ ]:
# --- Submission block ---------------------------------------------------
# Running this cell packages your answers for the auto-grader.
import json as _json
submission = {
    "student_id":        "FILL-IN",        # TODO: replace with your Coursera user id
    "perf":              perf,
    "peak_bytes":        peak_bytes,
    "mem_overhead_ratio": mem_overhead_ratio,
    "top3":              [(n, t) for (n, t) in top3],
    "landscape_row":     landscape_row,
}
with open("submission.json", "w") as _f:
    _json.dump(submission, _f, indent=2, default=str)
print("submission.json written. Download it and upload to the course portal.")


---

## Optional — Dataflow Deep Dive (Spyre hardware access required)

> *Only complete this section if you have been granted Spyre access through the IBM Research academic partnership.*

Open `../optional_spyre/lab1_spyre_extension.ipynb` and run the same benchmarking harness against Spyre hardware. Extend your `landscape_row` into a second row for Spyre, and write a **3rd analysis paragraph** comparing where Spyre's static dataflow model produces materially different numbers from your GPU run.

---

*Course: Dataflow Architectures for AI — Week 1 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*
